# `NumPy` 入门及实践

**学习目标：** 本节介绍 `NumPy` 数组、维度变换、广播、聚合、合并拆分、常用函数和线性代数，是科学计算的基础。


## `NumPy` 安装


通过如下命令安装 `NumPy`：

```bash
pip install numpy
```


## `NumPy` 介绍


`NumPy` 是开源的 `Python` 科学计算基础库，主要提供：

1. 强大的 `N` 维数组对象 `ndarray`。
2. 成熟的广播 `broadcasting` 机制。
3. 整合 `C`、`C++` 和 `Fortran` 代码的工具。
4. 线性代数、傅里叶变换和随机数生成等常用科学计算函数。

`NumPy` 是 `SciPy`、`Pandas`、`scikit-learn` 等数据处理和科学计算库的基础。许多 `NumPy` 函数底层使用 `C` 和 `BLAS` 实现，并可能链接到 `MKL` 或 `OpenBLAS`，因此在数值计算中通常比纯 `Python` 循环更高效。


In [ ]:
import numpy as np

rng = np.random.default_rng(42)

def compute_reciprocals(values):
    output = np.empty(len(values))
    for i in range(len(values)):
        output[i] = 1.0 / values[i]
    return output


In [ ]:
big_array = rng.integers(1, 100, size=1_000_000)


In [ ]:
%%timeit
compute_reciprocals(big_array)


上面的代码测试显式循环的耗时。下面测试 `NumPy` 向量化写法的耗时。


In [ ]:
%%timeit
1.0 / big_array


## 数组 `ndarray`


### 数组数组对象 `ndarray`


`Python` 已有列表类型，为什么还需要数组对象？

- 数组对象可以省去元素级运算中的显式循环，让一维向量更像一个整体。
- 数组对象针对数值计算做了优化，通常比纯 `Python` 列表更快。
- 科学计算中，同一维度的数据通常类型相同；统一数据类型有助于节省存储空间并提升计算效率。

`ndarray` 是 `NumPy` 的多维数组对象，由两部分构成：

- 实际数据。
- 描述数据的元数据，例如维度、形状和数据类型。

`ndarray` 通常要求所有元素类型相同，下标从 `0` 开始。


### `ndarray` 对象的属性


属性 | 说明
:---| :---
`.ndim ` |   秩或维度，即轴的数量或维度的数量
`.shape` |   ndarray 对象的尺度，对于矩阵，`n` 行 `m` 列
`.size ` |    ndarray 对象元素的个数，相当于 `.shape` 中 `n*m` 的值
`.dtype`  |    ndarray 对象的元素类型
`.itemsize` |    ndarray 对象中每个元素的大小，以字节为单位


ndarray 的维度即为对应数据所在的空间维度，1 维可以理解为直线空间，2 维可以理解为平面空间，3 维可以理解为立方体空间。


![NumPy 数组结构示意图](https://img-blog.csdnimg.cn/img_convert/0d82102231c202e4735ee3eb44504650.png)


轴 `axis` 用于描述多维数组的方向。按照数学惯例，可以用 $i$、$j$、$k$ 表示不同方向。

在一维空间中，一个轴即可描述位置，`NumPy` 记为 `axis=0`，空间内的数可以理解为离散点 $x_i$。

在二维空间中，需要两个轴，`NumPy` 记为 `axis=0` 和 `axis=1`，空间内的数可以理解为离散点 $(x_i, y_j)$。

在三维空间中，需要三个轴，`NumPy` 在二维基础上增加 `axis=2`，空间内的数可以理解为离散点 $(x_i, y_j, z_k)$。

在 `Python` 中，可以用 `ndim` 查看数组维度，用 `shape` 查看每个维度的长度。直观上也可以根据方括号 `[]` 的嵌套层数判断维度：有 $m$ 层就是 $m$ 维，最外层对应 `axis=0`，后续依次对应 `axis=1`、`axis=2`。


In [ ]:
a = np.array([1, 2, 3])
print(a.ndim)  # 一维数组。
print(a.shape)  # `axis 0` 的长度为 3。

b = np.array([[1, 2, 3], [4, 5, 6]])
print(b.ndim)  # 二维数组。
print(b.shape)  # shape=(2, 3)

c = np.array([[[1, 2, 3], [4, 5, 6]]])
print(c.ndim)  # 三维数组。
print(c.shape)  # shape=(1, 2, 3)


## 创建数组


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
# np 是 NumPy 惯用别名


### 从列表或元组导入
最常见的一种方法是传递一个列表或如同列表的对象，并以此通过 `numpy.array` 函数创建 `numpy` 数组。


`np.array()` 可以从列表或元组创建数组，也可以通过 `dtype` 指定数组元素类型。

```python
x = np.array(data)
x = np.array(data, dtype=np.float32)
```


In [ ]:
# list 转 ndarray
x = np.array([1, 2, 3, 4, 5])
print(x)

# tuple 转 ndarray
x = np.array((1, 2, 3, 4, 5))
print(x)

# 混合序列
x = np.array([[1, 2], [3, 4], (5, 6)])
print(x)


### `NumPy` 函数创建


函数|说明
:---|:---
`np.arange(n)`	|类似 `range()` 函数，返回 ndarray 类型，元素从 0 到 n‐1
`np.ones(shape)`	|根据 shape 生成一个全 1 数组，shape 是元组类型
`np.zeros(shape)`	|根据 shape 生成一个全 0 数组，shape 是元组类型
`np.full(shape,val)`	|根据 shape 生成一个数组，每个元素值都是 val
`np.eye(n)`	|创建一个正方的 n*n 单位矩阵，对角线为 1，其余为 0
`np.ones_like(a)`	|根据数组 a 的形状生成一个全 1 数组
`np.zeros_like(a)`|	根据数组 a 的形状生成一个全 0 数组
`np.full_like(a, val)`|	根据数组 a 的形状生成一个数组，每个元素值都是 val
`np.linspace()` | 根据起止数据等间距地填充数据，形成数组


`numpy.arange()` 和 `numpy.linspace()` 都可以创建等间隔数组。

```python
numpy.arange(start, stop, step, dtype=None)
numpy.linspace(start, stop, num=50, endpoint=True, retstep=False, dtype=None)
```


![`range()` 与 `linspace()` 对比](../../resources/images/linspace.png)

![`array_like` 示例](../../resources/images/array_like.png)


In [ ]:
# 区间 [1, 11)，步长 1
np.arange(1, 11, dtype='float')


In [ ]:
# 包含 stop
# 区间 [1, 10]
print(np.linspace(1, 10, 6, endpoint=True))
print(np.linspace(1, 10, 6, endpoint=False))


In [ ]:
# 不包含 stop
# 区间 [1, 10)
np.linspace(1, 11, 10, endpoint=False)


In [ ]:
np.zeros(1)


In [ ]:
# 2x3 矩阵，shape 用元组表示
print(np.zeros([2, 3]))
print(np.zeros((2, 3)))


### 其他方法


- `np.tile(A, reps)`：通过重复数组 `A` 构造新数组。
  - 如果 `reps` 的长度为 `d`，结果维度为 `max(d, A.ndim)`。
  - 如果 `A.ndim < d`，`A` 会通过添加新轴提升为 `d` 维。例如，形状 `(3,)` 会提升为 `(1, 3)` 以进行二维复制，或提升为 `(1, 1, 3)` 以进行三维复制。
  - 如果 `A.ndim > d`，`reps` 会在前面补 `1`。例如，形状为 `(2, 3, 4, 5)` 的数组遇到 `reps=(2, 2)` 时，会按 `reps=(1, 1, 2, 2)` 处理。
- `numpy.repeat(a, repeats, axis=None)`：沿指定轴重复数组元素。


![`tile()` 与 `repeat()` 对比](../../resources/images/np_repeat.png)


In [ ]:
a = np.array([1, 2, 3])
# 维度相同
print(np.tile(a, 2))

# a 升维为 (1, 3)
print(np.tile(a, (2, 2)))

# a 升维为 (1, 1, 3)
print(np.tile(a, (2, 1, 2)))

b = np.array([[1, 2], [3, 4]])
# reps 按 (1, 2) 处理
print(np.tile(b, 2))

# 维度相同
print(np.tile(b, (2, 1)))

# c 升维为 (1, 4)
c = np.array([1, 2, 3, 4])
print(np.tile(c, (4, 1)))


In [ ]:
a = [1, 2, 3]
# 整体重复
print('Tile:   ', np.tile(a, 2))

# 逐元素重复
print('Repeat: ', np.repeat(a, 2))


`random` 模块提供了很好的方法，可以生成任意给定 shape 的随机数（包括统计学中的随机分布）。


![NumPy 随机数生成示例](../../resources/images/np_rand.png)


新的随机数生成方法


![新版 NumPy 随机数生成示例](../../resources/images/np_rng.png)


## 数组变换


### 维度变换


| 方法 | 说明 |
| :--- | :--- |
| `reshape(shape)` | 不改变原数组，返回形状为 `shape` 的新视图或副本。 |
| `resize(shape)` | 修改原数组形状。 |
| `flatten()` | 将数组展平成一维数组，返回深拷贝。 |
| `ravel()` | 将数组展平成一维数组，通常返回浅拷贝或视图。 |
| `swapaxes(ax1, ax2)` | 交换数组的两个轴。 |
| `transpose(axes=None)` | 按指定轴顺序转置数组。 |


![NumPy 数组重塑与展平](../../resources/images/np_reshape.png)


In [ ]:
a = np.arange(24)
print(a)


In [ ]:
print(a.reshape(2, 3, 4))


In [ ]:
a.resize(2, 3, 4)
print(a)


In [ ]:
a.swapaxes(0, 1)


In [ ]:
a = np.array([(1, 2, 3, 4), (3, 1, 4, 2)])
b = a.flatten()
print(a)
print(b)


In [ ]:
a = np.array([(1, 2, 3, 4), (3, 1, 4, 2)])
b = a.ravel()
print(a)
print(b)


In [ ]:
a = np.arange(2*3*4).reshape(2, 3, 4) # 2x3x4
print(a)


In [ ]:
b = a.swapaxes(0, 2) # 4x3x2
print(b)


### 类型变换


可以通过 `dtype` 参数指定数组数据类型。常用的 `NumPy` 数据类型包括 `float`、`int`、`bool`、`str` 和 `object`。

为了更精确地控制内存占用，也可以使用 `float32`、`float64`、`int8`、`int16`、`int32` 等具体类型。


![NumPy 数据类型](../../resources/images/np_dtypes.png)


In [ ]:
# 二维浮点数组
list2d = [[0, 1, 2],[3, 4, 5],[6, 7, 8]]
arr2d_f = np.array(list2d, dtype=np.float32)
print(arr2d_f)
print(arr2d_f.dtype)


每个数字后面的小数点表示浮点数据类型，还可以使用 astype 方法将其转换为不同的数据类型。


In [ ]:
# 转为 int
print(arr2d_f)
arr2d_i = arr2d_f.astype('int')

print(arr2d_i)
print(arr2d_i.dtype)


In [ ]:
# 先转 int 再转 str
arr2d_s = arr2d_f.astype('int').astype('str')
print(arr2d_s)


一个 numpy 数组必须具有相同数据类型的所有条目，这是与列表相比，另一个显著的区别。

但是，如果不确定数组将持有何种数据类型，或者想要在同一个数组中共存字符和数字，可以将 `dtype` 设置为 `object`。


In [ ]:
# 布尔数组
arr2d_b = np.array([1, 0, 10, -1, None, False, ''], dtype='bool')
print(arr2d_b)


In [ ]:
arr1d_obj = np.array([1, 'a'], dtype='object')
print(type(arr1d_obj))
print(arr1d_obj)


可以通过 `tolist()` 方法，将 `NumPy` 的 `ndarray` 对象转换回 `Python` 的 `list` 对象。


In [ ]:
print(type(arr1d_obj.tolist()))


## 数组操作


### 数组的索引和切片


- 索引：获取数组中特定位置元素的过程。
- 切片：获取数组元素子集的过程。


![NumPy 索引示例](../../resources/images/np_index.png)


In [ ]:
a = np.arange(1, 6)
b = a[2:4]
print(a, b)
b[0] = 10
print(a, b)


In [ ]:
a = np.arange(1, 6)
b = a[[2, 3]]
print(a, b)
b[0] = 10
print(a, b)


***
将索引放入一个列表中，变成 fancy indexing，可以获得数组的拷贝，而不是视图！
***


`NumPy` 数组切片返回的是视图 `view`，这与 `Python` 列表切片通常返回新列表不同。


![NumPy 与列表索引对比](../../resources/images/np_vs_list.png)


另一种从 NumPy 数组获取数据的超级有用的方法是布尔索引，它允许使用各种类型的逻辑运算符。


![NumPy 布尔索引示例](../../resources/images/np_bool.png)


### `NumPy` 数组广播机制


广播 `Broadcasting` 描述了 `NumPy` 在算术运算中如何处理不同形状的数组。在满足兼容规则时，较小的数组会被广播到较大数组的形状，从而完成向量化计算。

广播可以避免显式 `Python` 循环，也能减少不必要的数据复制。最简单的情况是两个数组形状完全相同，此时运算会逐元素执行。


In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([2.0, 2.0, 2.0])
print(a*b)


当数组的形状满足某些约束时，`NumPy` 的广播规则放宽了这种约束。当一个数组和一个标量值在一个操作中组合时，会发生最简单的广播。


In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = 2.0
print(a*b) # 可以想象b自动扩充为[2.0, 2.0, 2.0]


### 广播的一般规则
广播机制首先需要判断参与计算的两个数组能否被广播机制处理，即判断是否广播兼容，规则是，比较两个数组的 `shape`，从 `shape` 的尾部开始一一比对。

1. 如果两个数组的维度相同，对应位置上轴的长度相同或其中一个的轴长度为 1，广播兼容，可在轴长度为 1 的轴上进行广播机制处理。

2. 如果两个数组的维度不同，那么给低维度的数组前扩展提升一维，扩展维的轴长度为 1，然后在扩展出的维上进行广播机制处理。


1. 两个数组各维度大小从后往前比对均一致。


In [ ]:
a = np.arange(1, 16).reshape([3, 5])
print(a)
print(a.shape, a.ndim)
b = np.array([2, 3, 4, 5, 6])
print(b)
print(b.shape, b.ndim)
print(a + b)


上面的例子中，`a` 数组的维度是 2 即二维数组 `(3, 5)`，`b` 是一维数组 `(5, )`。`b` 的最后一维的轴长度为 5、`a` 的最后一维的轴长度 5，满足从后向前开始对应轴的轴长度相同规则。两个数组的维度不同一个是 2 维一个是 1 维，那么对 `b` 前扩展增加维度变为 `(1, 5)` 成为 2 维数组，此时 `a` 和 `b` 都是二维数组，维度相同。增维后的 `b` 的倒数第二维的长度是 1，`a` 倒数第二维轴长度为 3，满足维度相同某轴长度为 1 的兼容规则，继续广播机制，将 `b` 倒数第二维轴长度继续加 1 直至变为和 `a` 数组倒数第二轴轴长度相同，这是 `b` 的 `shape` 经过不断的调整（广播机制）变为 `(3, 5)`，和 `a` 的 `shape` 相同，就可以广播计算了。每次对 `b` 倒数第二轴轴长度加 1 实际是拷贝增加一行。


2. 两个数组存在一些维度大小不相等时，有一个数组的该不相等维度大小为 1。


In [ ]:
A = np.zeros((2, 3, 4))
B = np.zeros((3, 1))
print((A+B).shape)  # 输出：(2, 3, 4)

A = np.zeros((2, 3, 4))
B = np.zeros((2, 1, 4))
print((A+B).shape)  # 输出：(2, 3, 4)

A = np.zeros((1))
B = np.zeros((3, 4))
print((A+B).shape)  # 输出(3, 4)


数组与标量之间的运算。标量维度 `(0, )`，先提升为 `(1, )`，在按广播规则处理。


In [ ]:
a = np.arange(24).reshape((2, 3, 4))
print("a:\n", a)

print("a+1:\n", a + 1)


> 下面这种写法也会触发广播，但它会直接修改原数组，副作用很大。实际项目中应先确认是否需要原地修改。


In [ ]:
a = np.arange(1, 5)
b = np.arange(4, 8).reshape(4, 1)
print(a)
print(b)
print(a + b)


### 对 `ndarray` 中的数据执行元素级运算的函数


函数 | 说明
:--- | :---
`np.abs(x) np.fabs(x)` |	计算数组各元素的绝对值
`np.sqrt(x)`|	计算数组各元素的平方根
`np.square(x)`|	计算数组各元素的平方
`np.log(x) np.log10(x) np.log2(x)`	|计算数组各元素的自然对数、10 底对数和 2 底对数
`np.ceil(x) np.floor(x)`	|计算数组各元素的 ceiling 值 或 floor 值
`np.rint(x)`	|计算数组各元素的四舍五入值
`np.modf(x)`	|将数组各元素的小数和整数部分以两个独立数组形式返回
`np.cos(x) np.cosh(x) np.sin(x) np.sinh(x) np.tan(x) np.tanh(x)`	|计算数组各元素的普通型和双曲型三角函数
`np.exp(x)`	|计算数组各元素的指数值
`np.sign(x)`	|计算数组各元素的符号值，1（+）, 0, ‐1（‐）


例子：计算神经网络中 `softmax` 层输出。假设 $z = [3, 1, -3]$。

![`softmax` 层示意图](../../resources/images/softmax.png)


In [ ]:
x = np.array([3, 1, -3], dtype=np.float32)
print(x)
print(np.exp(x))
print(np.exp(x) / sum(np.exp(x)))


### 聚合操作


所谓聚合操作，就是：求最大值、最小值、求和、均值、方差、中位数。


一维数组的聚合操作


![NumPy 一维数组聚合操作](../../resources/images/np_agg_1d.png)


In [ ]:
a = np.array([1, 2, 3])
print(a.max())
print(np.max(a))
print(np.percentile(a, q=[0, 25, 50, 75, 100]))


矩阵的聚合函数


![NumPy 矩阵聚合操作示例 1](../../resources/images/np_matrix_agg_1.png)


不仅可以聚合矩阵中的所有值，还可以通过使用 `axis` 参数跨行或列聚合


![NumPy 矩阵聚合操作示例 2](../../resources/images/np_matrix_agg_2.png)


下表提供了 `NumPy` 中可用的实用聚合函数的列表：

|函数名称 |	`NaN` 安全的版本|	描述|
|:--|:--|:--|
`np.sum` |	`np.nansum` |	计算元素的和
`np.prod` |	`np.nanprod` |	计算元素的积
`np.mean` |	`np.nanmean` |	计算元素的均值
`np.std` |	`np.nanstd` |	计算标准差
`np.var` |	`np.nanvar` |	计算方差
`np.min`  |	`np.nanmin` |	寻找最小值
`np.max`  |	`np.nanmax` |	寻找最大值
`np.argmin`  |	`np.nanargmin` |	寻找最小值的下标
`np.argmax`  |	`np.nanargmax` |	寻找最大值的下标
`np.median`  |	`np.nanmedian` |	计算元素的中值
`np.percentile` |	`np.nanpercentile` |	计算元素的百分位数
`np.any `|	`N/A` |	计算是否任何元素是真
`np.all` |	`N/A` |	计算是否所有元素是真


***
注意：argmin 和 argmax 函数用于二维及以上的数组是，返回的是扁平化之后的索引。可以用 `unravel_index` 转换。
***


![NumPy 矩阵 `argmin()` 示例](../../resources/images/matrix_argmin.png)


`np.any, np.all` 也要注意 `axis` 参数。


![NumPy `any()` 示例](../../resources/images/np_any.png)


### `NumPy` 数组合并与拆分


### 合并


`NumPy` 提供了 `concatenate()`、`append()`、`stack()`、`hstack()`、`vstack()`、`dstack()`、`row_stack()`、`column_stack()`、`r_` 和 `c_` 等函数或辅助对象，用于完成数组拼接操作。


方法 | 说明
:--- | :---
`concatenate` |提供了 `axis` 参数，用于指定拼接方向
`append`	| 默认先 ravel 再拼接成一维数组，也可指定 `axis`
`stack`  |	提供了 `axis` 参数，用于生成新的维度
`hstack`	|水平拼接，沿着行的方向，对列进行拼接
`vstack`	|垂直拼接，沿着列的方向，对行进行拼接
`dstack`	|沿着第三个轴（深度方向）进行拼接
`column_stack`	|水平拼接，沿着行的方向，对列进行拼接
`row_stack`	|垂直拼接，沿着列的方向，对行进行拼接
`r_`	|垂直拼接，沿着列的方向，对行进行拼接
`c_`	|水平拼接，沿着行的方向，对列进行拼接


### `numpy.concatenate()`

`numpy.concatenate()` 用于沿指定轴拼接多个数组。

函数签名：

```python
numpy.concatenate((a1, a2, ...), axis=0, out=None, dtype=None, casting="same_kind")
```

参数说明：

- `a1`、`a2`、`...`：待拼接数组组成的元组或列表。除拼接轴外，其余维度必须具有相同形状。
- `axis`：拼接方向，默认值为 `0`。当 `axis=0` 时沿第一个轴拼接。
- `out`：可选输出数组。若提供，形状必须与拼接结果一致。
- `dtype`：可选目标数据类型。不能与 `out` 同时提供。
- `casting`：控制允许的数据类型转换方式，默认值为 `"same_kind"`。


1. 如果拼接两个矩阵，可以只使用 hstack 和 vstack:


![NumPy 垂直和水平堆叠示例](../../resources/images/np_vhstack.png)


In [ ]:
a = np.arange(1, 13).reshape(3, 4)
b = np.arange(1, 9).reshape(2, 4)
c = np.arange(1, 7).reshape(3, 2)
print(np.hstack((a, c)))
print(np.concatenate((a, c), axis=1))
print(np.vstack((a, b)))
print(np.concatenate((a, b), axis=0))


2. 如果你在拼接矩阵和向量，使用 `hstack()` 会变得很麻烦，所以 `column_stack()` 是更好的选择：

![NumPy `hstack()` 示例](../../resources/images/np_hstack.png)


In [ ]:
a = np.arange(1, 13).reshape(3, 4)
b = np.array([1, 2, 3, 4])
c = np.array([1, 3, 5])
print(np.vstack((a, b)))
print(np.hstack((a, c[:,None])))
print(np.hstack((a, c.reshape(-1, 1))))
print(np.column_stack((a, c)))


3. `numpy.stack(arrays, axis=0, out=None)`

`numpy.stack()` 会沿指定 `axis` 对多个数组进行拼接，要求 `arrays` 中每个数组的形状一致。返回结果会比原数组多一个维度。`axis` 默认值为 `0`；当 `axis=-1` 时，会沿最后一个新轴堆叠。


In [ ]:
a = np.arange(1, 7).reshape(2, 3)
b = np.arange(7, 13).reshape(2, 3)
print("a: ")
print(a)
print("b:")
print(b)
print("a+b axis 0:")
print(np.stack((a, b)))  # 新增 axis=0
print("a+b axis 1:")
print(np.stack((a, b), axis=1))  # 新增 axis=1
print("a+b axis 2:")
# 新增 axis=2
print(np.stack((a, b), axis=2))


4. `r_` 和 `c_`


In [ ]:
print(np.r_[a, b])  # 垂直拼接，沿着列的方向，对行进行拼接
print(np.c_[a, b])  # 水平拼接，沿着行的方向，对列进行拼接


5. `numpy.append(arr, values, axis=None)`

 - `arr`：数组
 - `values`: array_like 的数据，若 `axis` 为 `None`，则先将 `arr` 和 `values` 进行 ravel 扁平化，再拼接；否则 `values` 应当与 `arr` 的 shape 一致，或至多在拼接 `axis` 的方向不一致。
 - `axis`：进行 `append` 操作的 `axis` 的方向，默认 `None`。


![NumPy `append()` 示例](../../resources/images/np_append.png)


In [ ]:
a = np.array([[1, 2, 3], [4, 5, 6]])
b = np.array([[7, 8, 9], [11, 12, 13]])
print(np.append(a, b))  # 先ravel扁平化再拼接，所以返回值为一个1维数组
print(np.append(a, b, axis=0))  # 沿第一个轴拼接，这里为行的方向
print(np.append(a, b, axis=1))  # 沿第二个轴拼接，这里为列的方向


***
一般不采用 append 操作。如果需要在数组的边界插入数据，可以采用 `numpy.pad`。
***


6. `numpy.pad(array, pad_width, mode='constant', **kwargs)`


- `array`：需要填充的数组。
- `pad_width`：每个轴 `axis` 边缘需要填充的数值数量。参数形式为 `((before_1, after_1), ..., (before_N, after_N))`。
- `mode`：填充方式，共有多种模式：
  - `'constant'`：使用常量填充。通过 `constant_values=(x, y)` 可以分别指定前后填充值。
  - `'edge'`：使用边缘值填充。
  - `'linear_ramp'`：使用边缘递减方式填充。
  - `'maximum'`：使用最大值填充。
  - `'mean'`：使用均值填充。
  - `'median'`：使用中位数填充。
  - `'minimum'`：使用最小值填充。
  - `'reflect'`：镜像填充。
  - `'symmetric'`：对称填充。
  - `'wrap'`：用原数组后面的值填充前面，前面的值填充后面。


![NumPy `pad()` 示例](../../resources/images/np_pad.png)


In [ ]:
a = np.arange(1, 16).reshape(3, -1)
np.pad(a, 1, mode='edge')


### 拆分


与合并相反的操作是拆分（split）。


![NumPy `stack()` 示例](../../resources/images/np_stack.png)


所有的拆分操作要么输入一组要拆分的索引，要么输入一个整数，即大小相等的子数组个数。


![NumPy `split()` 示例](../../resources/images/np_split.png)


### 插入与删除


`numpy.insert()` 用于在指定位置插入元素。

函数签名：

```python
numpy.insert(arr, obj, values, axis=None)
```

参数说明：

- `arr`：需要处理的数组。
- `obj`：一个或多个插入位置索引。
- `values`：要插入的值。如果 `values` 的类型与 `arr` 不同，`NumPy` 会尝试将其转换为 `arr` 的数据类型。
- `axis`：插入方向。`axis=None` 时会先将 `arr` 展平；`axis=0` 表示按行插入；`axis=1` 表示按列插入。


`numpy.delete()` 用于删除数组中指定位置的元素。

函数签名：

```python
numpy.delete(arr, obj, axis=None)
```

参数说明：

- `arr`：需要处理的数组。
- `obj`：一个或多个待删除位置索引。
- `axis`：删除方向。`axis=None` 时会先将 `arr` 展平；`axis=0` 表示按行删除；`axis=1` 表示按列删除。


![NumPy `insert()` 示例](../../resources/images/np_insert.png)


![NumPy `delete()` 示例](../../resources/images/np_delete.png)


### 练习


### 第 1 题

创建一个 `m x n` 二维数组，使其元素满足 $a_{ij} = j - i$。

例如，当 `m = 2`、`n = 3` 时，输出：

```python
[[ 0,  1,  2],
 [-1,  0,  1]]
```


In [ ]:
m = 2
n = 3
a = np.zeros((m, n))
for i in range(m):
    for j in range(n):
        a[i, j] = j - i
print(a)


In [ ]:
print(np.array([j - i for i in range(m) for j in range(n) ]).reshape(m, n))


In [ ]:
print(np.arange(n) - np.arange(m).reshape(-1, 1))


In [ ]:
i, j = np.meshgrid(np.arange(m), np.arange(n), indexing='ij')
print(j-i)


### 第 2 题

给定一个一维数组 `x`，编写函数计算 $f(x)$ 的值。

$$
f(x) = x^2 + x + 1
$$

例如，`x = [1, 2, 3, 4, 5]` 时，`f(x) = [3, 7, 13, 21, 31]`。


In [ ]:
def f(x):
    return x**2 + x + 1

x = [1, 2, 3, 4, 5]
print(list(map(f, x)))


In [ ]:
print(np.vectorize(f)(x).tolist())


In [ ]:
print(np.vectorize(lambda x:x**2 + x + 1)(x).tolist())


### 第 3 题

给定一维数组 `x`，编写函数计算分段函数 $f(x)$ 的值。

$$
f(x) =
\begin{cases}
1 + x, & x < 0 \\
1 - x, & x \ge 0
\end{cases}
$$

例如，`x = [-1, -2, 0, 1, 2]` 时，`f(x) = [0, -1, 1, 0, -1]`。


In [ ]:
def f(x):
    if x < 0:
        return 1 + x
    else:
        return 1 - x

x = [-1, -2, 0, 1, 2]
print(list(map(f, x)))


In [ ]:
print(np.vectorize(f)(x).tolist())


In [ ]:
x = np.array([-1, -2, 0, 1, 2])
np.where(x >= 0, 1-x, 1+x)


In [ ]:
x = np.array([-1, -2, 0, 1, 2])
np.select([x >= 0, x < 0], [1-x, 1+x], x)


In [ ]:
x = np.array([-1, -2, 0, 1, 2])

np.piecewise(x, [x >= 0, x < 0], [lambda x:1-x, lambda x:1+x])


### 第 4 题

给定二维数组，计算每一行的最大值与最小值之差，以及每一列最大值与最小值之差。

示例输入：

```python
a = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
])
```

期望输出：

```python
[3, 3, 3]
[8, 8, 8, 8]
```


In [ ]:
a = np.array([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])
row_diff = np.zeros(a.shape[0])
col_diff = np.zeros(a.shape[1])
for i in range(a.shape[0]):
    row_diff[i] = a[i, :].max() - a[i, :].min()

for j in range(a.shape[1]):
    col_diff[j] = a[:, j].max() - a[:, j].min()

print(row_diff)
print(col_diff)


In [ ]:
print(np.apply_along_axis(lambda x:x.max()-x.min(), axis=1, arr=a))
print(np.apply_along_axis(lambda x:x.max()-x.min(), axis=0, arr=a))


### 第 5 题


给定一个 `m x n` 的二维数组，每一行表示一个 `n` 维点，所以总共有 `m` 个点。现在给定一个点 `p` 和阈值 `eps`，返回所有到 `p` 的距离小于等于 `eps` 的点的平均值，也就是这些点的质心。

示例输入：

```python
a = np.array([
    [2, 1],
    [1, 6],
    [1, 7],
    [7, 1],
    [9, 4],
])
p = np.array([3, 3])
eps = 5
```

期望输出：

```python
[2.75, 3.75]
```


In [ ]:
a = np.array([[2, 1], [1, 6], [1, 7], [7, 1], [9, 4]])
p = np.array([3, 3])
eps = 5

sp = np.zeros_like(p)
cnts = 0
for pi in a:
    if np.linalg.norm(pi - p) <= eps:
        sp += pi
        cnts += 1

print(sp / cnts)


In [ ]:
a = np.array([[2, 1], [1, 6], [1, 7], [7, 1], [9, 4]])
p = np.array([3, 3])
eps = 5
a[np.linalg.norm(a - p, axis=1) <= eps, :].mean(axis=0)


### 第 6 题


给定 $d$ 维空间随机变量 $\boldsymbol{x}$ 的概率密度函数：

$$
p(\boldsymbol{x}) = \frac{1}{(2\pi)^{d / 2}|\boldsymbol{\Sigma}|^{1 / 2}}
\exp\left(-\frac{1}{2}(\boldsymbol{x} - \boldsymbol{\mu})^{\mathrm{T}}\boldsymbol{\Sigma}^{-1}(\boldsymbol{x} - \boldsymbol{\mu})\right)
$$

现在输入 $\boldsymbol{x}$、$\boldsymbol{\mu}$ 和 $\boldsymbol{\Sigma}$，输出对应概率密度。

示例输入：

```python
X = np.array([
    [2, 1],
    [1, 3],
    [3, 3],
])
mu = np.array([0, 0])
sigma = np.array([
    [1, 0],
    [0, 1],
])
```

期望输出：

```python
[1.30642333e-02, 1.07237757e-03, 1.96412803e-05]
```


In [ ]:
def multi_gaussian(X, mu, sigma):
    def gaussian(x):
        return (1.0 / np.sqrt((2*np.pi)**sigma.shape[0] * np.linalg.det(sigma))) * np.exp(-0.5 * (x - mu) @ np.linalg.pinv(sigma) @ (x - mu).T)
    return np.apply_along_axis(gaussian, axis=1, arr=X)

X = np.array([[2, 1], [1, 3],[3, 3]])

mu = np.array([0, 0])
sigma = np.array([[1, 0], [0, 1]])
print(multi_gaussian(X, mu, sigma))


## `NumPy` 中重要的函数


### `numpy.meshgrids`


`numpy.meshgrid()` 用于根据一维坐标数组生成坐标网格。

函数签名：

```python
numpy.meshgrid(*xi, copy=True, sparse=False, indexing="xy")
```

参数说明：

- `xi`：一个或多个一维坐标数组，常见形式为 `x`、`y` 或 `x`、`y`、`z`。
- `copy`：是否复制输入数组。
- `sparse`：是否生成稀疏网格。
- `indexing`：控制坐标系方向。`"xy"` 返回笛卡尔坐标索引，`"ij"` 返回矩阵索引。二维情况下，输入长度为 `M` 和 `N` 时，`"xy"` 输出形状为 `(N, M)`，`"ij"` 输出形状为 `(M, N)`。


In [ ]:
x = np.linspace(1, 10, 5)
y = np.linspace(1, 10, 10)
xx, yy = np.meshgrid(x, y)
print(xx)
print(yy)


In [ ]:
i, j = np.meshgrid(np.arange(2), np.arange(3))
print(i)
print(j)


In [ ]:
i, j = np.meshgrid(np.arange(3), np.arange(2), indexing='ij')
print(i)
print(j)


如果为了仅仅是为了生成二维数组（矩阵）的索引，可以用 `np.indices` 替换实现同样的功能。


In [ ]:
i, j = np.indices((3, 2))
print(i)
print(j)


`np.indices` 可以用来切片二维数组的元素。


In [ ]:
a = np.arange(12).reshape(3, 4)
i, j = np.indices((2, 2))
print(a)
print(a[i, j])
print(a[:2, :2])  # 与上面等价
print(a[i+1, j+1])  # 适合提取固定尺寸的子数组


`np.meshgrid` 的一般应用场景，可以用来绘制三维图形，或者等高线。例如机器学习里面的 SVC 超平面。


In [ ]:
import matplotlib.pyplot as plt

x = np.linspace(-10, 10, 25)
y = np.linspace(-10, 10, 25)

xx, yy = np.meshgrid(x, y)
zz = xx ** 2 + 4 * yy ** 2

plt.contourf(xx, yy, zz, cmap="rainbow")


### `numpy.where`


在此之前，已经介绍了如何从满足给定条件的矩阵中提取 `item`, 即布尔检索。

但有时想知道项目的索引位置（满足某条件），并做一些处理，这是可以采用 `numpy.where`。

`numpy.where` 可以定位矩阵中一个给定条件为 `True` 的位置。


![NumPy `where()` 索引示例](../../resources/images/np_where.png)


需要注意，`np.where`、`np.nonzero` 等函数返回的是一个元组，如果需要获得数组，要写成 `np.where(a>5)[0]`。


In [ ]:
a = rng.integers(0, 10, size=10)

print("Array: ", a)

idex_gt5 = np.where(a > 5)

print("Positions where value > 5: ", idex_gt5)  # 元组
print("Positions where value > 5: ", idex_gt5[0])  # 数组
print("Positions where value > 5: ", idex_gt5[0].tolist())  # 列表


通过 `np.where` 获取的是满足条件的 `item` 的索引数组，一旦获取到这个索引数组，就可以通过 `take` 方法，获取对应矩阵的值


In [ ]:
print("items of values > 5: ", a.take(idex_gt5))


此外，`np.where` 也接受额外的可选参数：`x` 与 `y`。

只要条件为真，即得到 `x`，否则得到 `y`

下面的例子中，将尝试创建一个数组：大于 5 的位置将设置为 1；否则设置为 0


In [ ]:
print(a)
print(np.where(a > 5, 1, 0))  # 二值化


此外，可以通过 `np.argmax` 获得最大值所在的索引；`np.argmin` 获得最小值所在的索引


In [ ]:
print(a)
# 最大值位置
print('Position of max value: ', np.argmax(a))
print('Max value is: ', a[np.argmax(a)])

# 最小值位置
print('Position of min value: ', np.argmin(a))
print('Min value is: ', a[np.argmin(a)])


***
2D 及以上的 argmin 和 argmax 函数有一个麻烦，就是返回扁平化之后的索引。要将其转换为两个坐标，需要使用 `np.unravel_index` 函数。
***


In [ ]:
a = rng.integers(1, 10, (3, 4))
print(a)
print(np.argmin(a)) # 扁平索引
np.unravel_index(np.argmin(a), a.shape) # 多维索引


### `numpy.sort`


`NumPy` 提供了多种排序的方法。这些排序函数实现不同的排序算法，每个排序算法的特征在于执行速度，最坏情况性能，所需的工作空间和算法的稳定性。下表显示了三种排序算法的比较。

|种类 |	速度 |	最坏情况 | 工作空间	| 稳定性|
|:--:|:--:|:--:|:--:|:--:|
'quicksort' （快速排序）|	1	|`O(n^2)`|	0|	否
'mergesort'（归并排序）|	2	|O(n*`log(n)`)	|~`n/2`	|是
'heapsort'（堆排序）	|3	|O(n*`log(n)`)	|0	|否


`numpy.sort()` 用于对数组排序。

函数签名：

```python
numpy.sort(a, axis=-1, kind=None, order=None)
```

参数说明：

- `a`：要排序的数组。
- `axis`：排序轴。默认值为 `-1`，表示沿最后一个轴排序；`axis=None` 时会先展开数组。
- `kind`：排序算法，例如 `'quicksort'`、`'mergesort'`、`'heapsort'` 或 `'stable'`。
- `order`：当数组包含字段时，用于指定排序字段。


In [ ]:
arr = rng.integers(1, 6, size=[8, 4])
print(arr)


如果使用 `np.sort()` 并设置 `axis=0`，每一列会独立按升序排序。需要注意，这会重排每列内部数据，使同一行中不同列的原始对应关系不再保留。


In [ ]:
print(np.sort(arr, axis=0))


如果不希望破坏原有数据中行数据，建议使用间接的排序方法：`np.argsort`。


### `numpy.argsort`


`numpy.argsort()` 函数返回的是数组值从小到大的索引值。


In [ ]:
x = np.array([1, 10, 5, 2, 8, 9])
sort_index = np.argsort(x)
print('index positions of sorted array: \n')
print(sort_index)
print()
print('sorted array: \n')
print(x[sort_index])


现在，为了对初始数组:arr 排序，将要对第一列做一次 `argsort`，并且将获得的索引结果，尝试对 arr 进行排序：


In [ ]:
print(arr)
print('The 1st column data of arr: \n')
print(arr[:, 0])
print()
sorted_index_1stcol = arr[:, 0].argsort()
print('The sorted index of 1st column: \n')
print(sorted_index_1stcol)


In [ ]:
print('original arr is: \n')
print(arr)
print()
print('Sort arr by the result of sorted index of 1st column: \n')
print(arr[sorted_index_1stcol])


这是升序排列，那么如何降序排列呢？简单来说，仅仅翻转 sorted_index_1stcol 就可以了：


In [ ]:
print(arr[sorted_index_1stcol[::-1]])


### `numpy.lexsort`


`numpy.lexsort()` 用于对多个序列进行排序。可以把它理解为表格多列排序：每一列代表一个序列，排序时优先使用元组中最右侧的键。因此，传给 `np.lexsort()` 的排序键顺序要从次要键到主要键排列。


In [ ]:
print('arr \n')
print(arr)
print()
print('arr[:, 1] \n')
print(arr[:, 1])
print()
print('arr[:, 0] \n')
print(arr[:, 0])
print()
lexsorted_index = np.lexsort((arr[:,0], arr[:, 1], arr[:, 3]))
print('lexsorted_index \n')
print(lexsorted_index)
print()
print('sort arr by lextsorted_index: \n')
print(arr[lexsorted_index])
print()
print(arr[lexsorted_index[::-1]])


具体意思，就是先按照第一列进行排序，然后按照第二列进行排序。


### `NumPy` 函数编程


### `numpy.vectorize`


通过使用 `vectorize()`，可以把原本处理单个数字的函数转换为可处理数组的函数。


下面的函数 `foo()` 接受一个数字：如果数字是奇数，则返回它的平方；否则返回它除以 `2` 的结果。该函数可以处理标量，但不能直接处理数组。使用 `numpy.vectorize()` 可以把它包装成能够逐元素处理数组的函数。


In [ ]:
# 标量函数
def foo(x):
    return x**2 if x % 2 == 1 else x / 2

# 标量输入
print('x = 10 returns ', foo(10))
print('x = 11 returns ', foo(11))


In [ ]:
# 向量输入会失败
print('x = [10, 11, 12] returns ', foo([10, 11, 12]))  # 报错


下面将函数 `foo()` 向量化，使其可以接收数组参数：


In [ ]:
foo_v = np.vectorize(foo, otypes=[float])

print('x = [10, 11, 12] returns ', foo_v([10, 11, 12]))
print('x = [[10, 11, 12], [1, 2, 3]] returns ', foo_v([[10, 11, 12], [1, 2, 3]]))


当我们需要将作用于标量的函数，也适用于数组，vectorize 方法就非常有用了。


`numpy.vectorize()` 也接受可选参数 `otypes`，用于显式声明输出结果的数据类型。明确输出类型通常可以减少类型推断成本，让 `vectorize()` 运行得更稳定。


### `numpy.apply_along_axis()`

`numpy.apply_along_axis()` 用于沿指定轴把一维函数应用到数组切片上。

函数签名：

```python
numpy.apply_along_axis(func1d, axis, arr, *args, **kwargs)
```


参数：

- `fund1d` 作用于一维向量（）的函数。
-
- `axis` 的作用是应用: func1d。对于 2D 数组，1 是行，0 是列。
-
- `arr` 作用于参数 func1d 的数组。


In [ ]:
a = rng.integers(1, 10, size=(4, 5))
print(a)
# 按行应用
print('基于行轴, 所以数组成员为4个: ',
      np.apply_along_axis(lambda x : x.max() - x.min(), axis=1, arr=a))

# 按列应用
print('基于列轴, 所以数组成员为10个: ',
      np.apply_along_axis(lambda x : x.max() - x.min(), axis=0, arr=a))


### `numpy.piecewise()`

`numpy.piecewise()` 用于根据条件列表对数组的不同区域应用不同函数。

函数签名：

```python
numpy.piecewise(x, condlist, funclist, *args, **kw)
```

参数说明：

- `x`：要处理的数组。
- `condlist`：条件列表，可以由多个条件组成。
- `funclist`：函数列表，与 `condlist` 一一对应。某个条件为 `True` 时，执行对应函数。

返回值：

- 返回一个与输入 `x` 形状相同的数组。


`numpy.piecewise` 和 `where`、`select`、`choose` 实现的功能也有类似的，即根据相关的条件，进行筛选，然后对不同条件的元素进行相关的操作，这个操作可以来源与函数、lambda 表达式等，并得到新的结果。


In [ ]:
x = np.arange(0, 10)
print(x)
print(np.piecewise(x, [x < 4, x >= 6], [-1, 1]))
print(np.piecewise(x, [x < 4, x >= 6], [lambda x:x**2, lambda x:x*100]))


### `numpy.digitize`


使用 `np.digitize` 方法，可以返回每个元素所属的 bin 的索引位置。


`numpy.digitize()` 用于返回数组中每个值所属的区间索引。

函数签名：

```python
numpy.digitize(x, bins, right=False)
```

参数说明：

- `x`：输入数组。
- `bins`：一维单调数组，必须升序或降序排列。
- `right`：是否让区间包含右边界，默认值为 `False`。


In [ ]:
bins = np.array(range(-99, 102, 3))
a = np.digitize(-98, bins)  # a = 1
b = np.digitize(68, bins)  # b = 56
print(a)
print(b)


### `numpy.clip`


`numpy.clip()` 用于把数组中的值限制在给定范围内。

函数签名：

```python
numpy.clip(a, a_min, a_max, out=None)
```

参数说明：

- `a`：输入数组。
- `a_min`：下限。小于该值的元素会被替换为 `a_min`。
- `a_max`：上限。大于该值的元素会被替换为 `a_max`。
- `out`：可选输出数组。


In [ ]:
print(x)
# 限制到 [3, 8]
print(np.clip(x, 3, 8))
